# Quality SimFin

Value-level data quality checks: accounting identities, cross-statement consistency, plausibility.

Tolerance: relative diff > 0.5% (configurable in `irp.quality.rules.violations`).

In [1]:
import pandas as pd
from IPython.display import display
from pandas.io.formats.style_render import ExtFormatter

from irp.quality import REGISTRY, run

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 240)


def _edgar_link(url):
    return f'<a href="{url}" target="_blank">view</a>' if url else ''


_NUM_FMT: ExtFormatter = {
    'LHS_value': '{:,.0f}',
    'RHS_value': '{:,.0f}',
    'diff': '{:,.0f}',
    'rel_diff': '{:.2%}',
    'EDGAR': _edgar_link,
}

SAMPLE = ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'META', 'JPM', 'XOM', 'JNJ', 'WMT']

In [2]:
run(['VSAT']).style.format(_NUM_FMT)

,Rule,Statement,Ticker,Fiscal Year,Fiscal Period,Period,LHS_value,RHS_value,diff,rel_diff,Report Date,CIK,EDGAR
0,cash_chain,cashflow,VSAT,2024,FY,A,"-288,928,000","-292,759,000","3,831,000",1.31%,2025-03-31 00:00:00,797721,view
1,continuing_ops,income,VSAT,2024,FY,A,"-531,125,000","-544,353,000","13,228,000",2.43%,2025-03-31 00:00:00,797721,view
2,da_income_vs_cashflow,cross,VSAT,2024,FY,A,"263,933,000","1,360,807,000","-1,096,874,000",80.60%,2025-03-31 00:00:00,797721,view
3,ni_income_vs_cashflow,cross,VSAT,2024,FY,A,"-574,962,000","-531,125,000","-43,837,000",7.62%,2025-03-31 00:00:00,797721,view
4,continuing_ops,income,VSAT,2023,FY,A,"-1,047,497,000","-1,054,472,000","6,975,000",0.66%,2024-03-31 00:00:00,797721,view
5,da_income_vs_cashflow,cross,VSAT,2023,FY,A,"227,165,000","1,157,524,000","-930,359,000",80.37%,2024-03-31 00:00:00,797721,view
6,net_income_split,income,VSAT,2023,FY,A,"-1,068,904,000","-1,057,919,000","-10,985,000",1.03%,2024-03-31 00:00:00,797721,view
7,ni_income_vs_cashflow,cross,VSAT,2023,FY,A,"-1,068,904,000","-1,057,919,000","-10,985,000",1.03%,2024-03-31 00:00:00,797721,view
8,da_income_vs_cashflow,cross,VSAT,2022,FY,A,"29,811,000","500,377,000","-470,566,000",94.04%,2023-03-31 00:00:00,797721,view
9,net_income_split,income,VSAT,2022,FY,A,"1,084,806,000","1,090,748,000","-5,942,000",0.54%,2023-03-31 00:00:00,797721,view


## Registered Rules

In [3]:
_rules_df = pd.DataFrame([
    {'Rule': r.name, 'Statement': r.statement, 'LHS': r.lhs, 'RHS': r.rhs}
    for r in REGISTRY
])
display(_rules_df)

,Rule,Statement,LHS,RHS
0,assets_split,balance,Total Assets,Total Current Assets + Total Noncurrent Assets
1,liab_split,balance,Total Liabilities,Total Current Liabilities + Total Noncurrent L...
2,liab_equity_split,balance,Total Liabilities & Equity,Total Liabilities + Total Equity
3,accounting_equation,balance,Total Assets,Total Liabilities & Equity
4,gross_profit,income,Gross Profit,Revenue + Cost of Revenue
5,operating_income,income,Operating Income (Loss),Gross Profit + Operating Expenses
6,continuing_ops,income,Income (Loss) from Continuing Operations,Pretax Income (Loss) + Income Tax (Expense) Be...
7,net_income_split,income,Net Income,Income (Loss) from Continuing Operations + Net...
8,cash_chain,cashflow,Net Change in Cash,CFO + CFI + CFF
9,ni_income_vs_cashflow,cross,Net Income (income),Net Income/Starting Line (cashflow)


## Annual

In [4]:
findings_A = run(SAMPLE, 'A')
print(f'Total findings: {len(findings_A)}')
display(findings_A.drop(columns=['CIK'], errors='ignore').style.format(_NUM_FMT))

Total findings: 37


,Rule,Statement,Ticker,Fiscal Year,Fiscal Period,Period,LHS_value,RHS_value,diff,rel_diff,Report Date,EDGAR
0,cash_chain,cashflow,AMZN,2024,FY,A,"8,422,000,000","9,723,000,000","-1,301,000,000",13.38%,2024-12-31 00:00:00,view
1,cash_chain,cashflow,AMZN,2023,FY,A,"19,637,000,000","19,234,000,000","403,000,000",2.05%,2023-12-31 00:00:00,view
2,cash_chain,cashflow,AMZN,2022,FY,A,"17,776,000,000","18,869,000,000","-1,093,000,000",5.79%,2022-12-31 00:00:00,view
3,cash_chain,cashflow,AMZN,2021,FY,A,"-5,900,000,000","-5,536,000,000","-364,000,000",6.17%,2021-12-31 00:00:00,view
4,cash_chain,cashflow,AMZN,2020,FY,A,"5,967,000,000","5,349,000,000","618,000,000",10.36%,2020-12-31 00:00:00,view
5,cash_chain,cashflow,JNJ,2024,FY,A,"2,246,000,000","2,535,000,000","-289,000,000",11.40%,2024-12-31 00:00:00,view
6,cash_chain,cashflow,JNJ,2023,FY,A,"7,732,000,000","7,844,000,000","-112,000,000",1.43%,2023-12-31 00:00:00,view
7,cash_chain,cashflow,JNJ,2021,FY,A,"502,000,000","680,000,000","-178,000,000",26.18%,2021-12-31 00:00:00,view
8,cash_chain,cashflow,JNJ,2020,FY,A,"-3,320,000,000","-3,409,000,000","89,000,000",2.61%,2020-12-31 00:00:00,view
9,cash_chain,cashflow,META,2024,FY,A,"2,611,000,000","3,397,000,000","-786,000,000",23.14%,2024-12-31 00:00:00,view


## Quarterly

In [5]:
findings_Q = run(SAMPLE, 'Q')
print(f'Total findings: {len(findings_Q)}')
display(findings_Q.drop(columns=['CIK'], errors='ignore').style.format(_NUM_FMT))

Total findings: 147


,Rule,Statement,Ticker,Fiscal Year,Fiscal Period,Period,LHS_value,RHS_value,diff,rel_diff,Report Date,EDGAR
0,cash_chain,cashflow,AMZN,2025,Q1,Q,"-12,419,000,000","-12,835,000,000","416,000,000",3.24%,2025-03-31 00:00:00,view
1,cash_chain,cashflow,AMZN,2024,Q1,Q,"-558,000,000","-129,000,000","-429,000,000",76.88%,2024-03-31 00:00:00,view
2,cash_chain,cashflow,AMZN,2024,Q2,Q,"-1,659,000,000","-1,347,000,000","-312,000,000",18.81%,2024-06-30 00:00:00,view
3,cash_chain,cashflow,AMZN,2024,Q3,Q,"7,004,000,000","6,314,000,000","690,000,000",9.85%,2024-09-30 00:00:00,view
4,cash_chain,cashflow,AMZN,2024,Q4,Q,"3,635,000,000","4,885,000,000","-1,250,000,000",25.59%,2024-12-31 00:00:00,view
5,cash_chain,cashflow,AMZN,2023,Q1,Q,"-4,519,000,000","-4,664,000,000","145,000,000",3.11%,2023-03-31 00:00:00,view
6,cash_chain,cashflow,AMZN,2023,Q2,Q,"333,000,000","264,000,000","69,000,000",20.72%,2023-06-30 00:00:00,view
7,cash_chain,cashflow,AMZN,2023,Q3,Q,"14,000,000","516,000,000","-502,000,000",97.29%,2023-09-30 00:00:00,view
8,cash_chain,cashflow,AMZN,2023,Q4,Q,"23,809,000,000","23,118,000,000","691,000,000",2.90%,2023-12-31 00:00:00,view
9,cash_chain,cashflow,AMZN,2022,Q1,Q,"122,000,000","106,000,000","16,000,000",13.11%,2022-03-31 00:00:00,view


## Findings by Rule

In [6]:
if len(findings_A):
    display(findings_A.groupby(['Rule', 'Ticker']).size().unstack(fill_value=0))
else:
    print('No annual findings.')

Ticker,AMZN,JNJ,META,MSFT,WMT,XOM
Rule,,,,,,
cash_chain,5,4,4,5,5,4
ni_income_vs_cashflow,0,0,0,0,5,5


## Findings by Ticker

In [7]:
if len(findings_A):
    display(findings_A.groupby('Ticker').size().sort_values(ascending=False))
else:
    print('No annual findings.')

Ticker
WMT     10
XOM      9
MSFT     5
AMZN     5
JNJ      4
META     4
dtype: int64

## Detail per Rule

In [8]:
for _rule_name, _group in findings_A.groupby('Rule'):
    print(f'=== {_rule_name} ({len(_group)} violations) ===')
    display(_group.drop(columns=['Rule', 'Statement', 'CIK'], errors='ignore').style.format(_NUM_FMT))
    print()

=== cash_chain (27 violations) ===


,Ticker,Fiscal Year,Fiscal Period,Period,LHS_value,RHS_value,diff,rel_diff,Report Date,EDGAR
0,AMZN,2024,FY,A,"8,422,000,000","9,723,000,000","-1,301,000,000",13.38%,2024-12-31 00:00:00,view
1,AMZN,2023,FY,A,"19,637,000,000","19,234,000,000","403,000,000",2.05%,2023-12-31 00:00:00,view
2,AMZN,2022,FY,A,"17,776,000,000","18,869,000,000","-1,093,000,000",5.79%,2022-12-31 00:00:00,view
3,AMZN,2021,FY,A,"-5,900,000,000","-5,536,000,000","-364,000,000",6.17%,2021-12-31 00:00:00,view
4,AMZN,2020,FY,A,"5,967,000,000","5,349,000,000","618,000,000",10.36%,2020-12-31 00:00:00,view
5,JNJ,2024,FY,A,"2,246,000,000","2,535,000,000","-289,000,000",11.40%,2024-12-31 00:00:00,view
6,JNJ,2023,FY,A,"7,732,000,000","7,844,000,000","-112,000,000",1.43%,2023-12-31 00:00:00,view
7,JNJ,2021,FY,A,"502,000,000","680,000,000","-178,000,000",26.18%,2021-12-31 00:00:00,view
8,JNJ,2020,FY,A,"-3,320,000,000","-3,409,000,000","89,000,000",2.61%,2020-12-31 00:00:00,view
9,META,2024,FY,A,"2,611,000,000","3,397,000,000","-786,000,000",23.14%,2024-12-31 00:00:00,view



=== ni_income_vs_cashflow (10 violations) ===


,Ticker,Fiscal Year,Fiscal Period,Period,LHS_value,RHS_value,diff,rel_diff,Report Date,EDGAR
19,WMT,2024,FY,A,"19,436,000,000","20,157,000,000","-721,000,000",3.58%,2025-01-31 00:00:00,view
21,WMT,2023,FY,A,"15,511,000,000","16,270,000,000","-759,000,000",4.67%,2024-01-31 00:00:00,view
23,WMT,2022,FY,A,"11,680,000,000","11,292,000,000","388,000,000",3.32%,2023-01-31 00:00:00,view
25,WMT,2021,FY,A,"13,673,000,000","13,940,000,000","-267,000,000",1.92%,2022-01-31 00:00:00,view
27,WMT,2020,FY,A,"13,510,000,000","13,706,000,000","-196,000,000",1.43%,2021-01-31 00:00:00,view
29,XOM,2024,FY,A,"33,680,000,000","35,063,000,000","-1,383,000,000",3.94%,2024-12-31 00:00:00,view
31,XOM,2023,FY,A,"36,010,000,000","37,354,000,000","-1,344,000,000",3.60%,2023-12-31 00:00:00,view
32,XOM,2022,FY,A,"55,740,000,000","57,577,000,000","-1,837,000,000",3.19%,2022-12-31 00:00:00,view
34,XOM,2021,FY,A,"23,040,000,000","23,598,000,000","-558,000,000",2.36%,2021-12-31 00:00:00,view
36,XOM,2020,FY,A,"-22,440,000,000","-23,251,000,000","811,000,000",3.49%,2020-12-31 00:00:00,view
